# Satellite Acquisition Date Selection

Planet SuperDove imagery is acquired on-demand and has a cost per scene. This notebook identifies candidate date windows where the surface turbidity regime (low / medium / high / very high) is stable enough and sufficiently diverse relative to the existing archive to justify ordering additional acquisitions.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

In [2]:
CTD_DIR = Path("../data/raw/UPCT")
SAT_DIR = Path("../data/satellite")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

REGIME_THRESHOLDS = {"low": 5.0, "medium": 15.0, "high": 30.0}
REGIME_ORDER = ["low", "medium", "high", "very_high"]
REGIME_COLORS = {"low": "#2196F3", "medium": "#4CAF50", "high": "#FF9800", "very_high": "#F44336"}

ANALYSIS_START = "2022-01-01"
STABILITY_WINDOW = 14
MIN_STATION_COVERAGE = 0.5

## 1. Load surface turbidity across all CTD stations

Each file contains daily depth profiles. The shallowest measurement (`0.5 m`) is used as the surface proxy, since satellite reflectance integrates the topmost water column.

In [3]:
def load_surface_turbidity(path: Path) -> pd.Series:
    df = pd.read_csv(path, parse_dates=["Date"])
    df = df.dropna(subset=["Date"]).set_index("Date").sort_index()
    depth_cols = sorted([c for c in df.columns if c not in ("Date",)],
                        key=lambda x: float(x))
    surface_col = depth_cols[0]
    return pd.to_numeric(df[surface_col], errors="coerce").rename(path.stem)

turbidity_files = sorted(CTD_DIR.glob("CTD*_turbidity.csv"))
series_list = [load_surface_turbidity(f) for f in turbidity_files]

surface = pd.concat(series_list, axis=1)
surface = surface[surface.index >= ANALYSIS_START].sort_index()

print(f"Stations: {surface.shape[1]}  |  Dates: {surface.shape[0]}  |  Range: {surface.index[0].date()} – {surface.index[-1].date()}")

ValueError: could not convert string to float: 'Unnamed: 7'

## 2. Multi-station daily summary

Median across stations is the primary signal (robust to individual sensor failures). Station coverage fraction flags days with sparse data.

In [ ]:
daily = pd.DataFrame(index=surface.index)
daily["median_ntu"] = surface.median(axis=1)
daily["q25_ntu"]   = surface.quantile(0.25, axis=1)
daily["q75_ntu"]   = surface.quantile(0.75, axis=1)
daily["coverage"]  = surface.notna().mean(axis=1)

daily = daily[daily["coverage"] >= MIN_STATION_COVERAGE].copy()

print(f"Days with ≥{MIN_STATION_COVERAGE:.0%} station coverage: {len(daily)}")

## 3. Turbidity regime classification

Fixed NTU thresholds divide the dynamic range into four regimes that reflect optically distinct water states in Mar Menor.

In [ ]:
def assign_regime(ntu: float) -> str:
    if ntu < REGIME_THRESHOLDS["low"]:
        return "low"
    elif ntu < REGIME_THRESHOLDS["medium"]:
        return "medium"
    elif ntu < REGIME_THRESHOLDS["high"]:
        return "high"
    return "very_high"

daily["regime"] = daily["median_ntu"].apply(assign_regime)
daily["regime"] = pd.Categorical(daily["regime"], categories=REGIME_ORDER, ordered=True)

print(daily["regime"].value_counts().reindex(REGIME_ORDER))

## 4. Rolling stability index

The coefficient of variation (CV) over a rolling `STABILITY_WINDOW`-day window measures how predictable turbidity is within a short period. Low CV windows are ideal acquisition targets because the turbidity state is unlikely to change between the order being placed and the satellite overpass.

In [ ]:
roll = daily["median_ntu"].rolling(STABILITY_WINDOW, center=True, min_periods=STABILITY_WINDOW // 2)
daily["roll_mean"] = roll.mean()
daily["roll_std"]  = roll.std()
daily["roll_cv"]   = (daily["roll_std"] / daily["roll_mean"].replace(0, np.nan)).clip(upper=2.0)

cv_p25 = daily["roll_cv"].quantile(0.25)
daily["stable"] = daily["roll_cv"] <= cv_p25

print(f"Stable days (CV ≤ {cv_p25:.2f}): {daily['stable'].sum()}")

## 5. Existing satellite acquisition coverage

In [ ]:
sat_dates = pd.to_datetime(sorted({p.name.split("_")[0] for p in SAT_DIR.glob("*.tif")}))
sat_dates = sat_dates[sat_dates >= ANALYSIS_START]

sat_coverage = daily.reindex(sat_dates).dropna(subset=["median_ntu"])

print(f"Satellite dates in archive: {len(sat_dates)}")
print(f"Satellite dates with CTD data: {len(sat_coverage)}")
print("\nRegime distribution in archive:")
print(sat_coverage["regime"].value_counts().reindex(REGIME_ORDER))

## 6. Coverage gap analysis

Compares the regime distribution in the full CTD time series against the subset already covered by acquired imagery. Regimes with low relative coverage are under-represented in the training set.

In [ ]:
total_counts = daily["regime"].value_counts().reindex(REGIME_ORDER).fillna(0)
sat_counts   = sat_coverage["regime"].value_counts().reindex(REGIME_ORDER).fillna(0)
coverage_gap = pd.DataFrame({
    "total_days":     total_counts,
    "acquired_days":  sat_counts,
    "coverage_pct":   (sat_counts / total_counts * 100).round(1),
}).fillna(0)

print(coverage_gap.to_string())

## 7. Candidate window detection

Scans the time series for consecutive stable days belonging to under-represented regimes. A window qualifies if:
- regime coverage is below the median across all regimes, **and**
- rolling CV is below the 25th-percentile threshold (stable), **and**
- no satellite image already exists within the window.

In [ ]:
median_coverage = coverage_gap["coverage_pct"].median()
underrepresented = set(coverage_gap[coverage_gap["coverage_pct"] < median_coverage].index.tolist())

daily["acquired"] = daily.index.isin(sat_dates)
daily["candidate"] = daily["stable"] & daily["regime"].isin(underrepresented) & ~daily["acquired"]

windows = []
in_window = False
for date, row in daily.iterrows():
    if row["candidate"] and not in_window:
        w_start = date
        in_window = True
    elif not row["candidate"] and in_window:
        windows.append({"start": w_start, "end": date - pd.Timedelta(days=1)})
        in_window = False
if in_window:
    windows.append({"start": w_start, "end": daily.index[-1]})

if windows:
    win_df = pd.DataFrame(windows)
    win_df["length_days"] = (win_df["end"] - win_df["start"]).dt.days + 1
    win_df["regime"] = win_df.apply(
        lambda r: daily.loc[r["start"]:r["end"], "regime"].mode()[0], axis=1
    )
    win_df["median_ntu"] = win_df.apply(
        lambda r: daily.loc[r["start"]:r["end"], "median_ntu"].median().round(2), axis=1
    )
    win_df["cv"] = win_df.apply(
        lambda r: daily.loc[r["start"]:r["end"], "roll_cv"].median().round(3), axis=1
    )
    win_df = win_df[win_df["length_days"] >= 7].sort_values(["regime", "cv"]).reset_index(drop=True)
    print(f"Candidate windows (≥7 days, stable, under-represented): {len(win_df)}")
    print(win_df[["start", "end", "length_days", "regime", "median_ntu", "cv"]].to_string(index=False))
else:
    win_df = pd.DataFrame()
    print("No candidate windows found.")

## 8. Visualisation

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

ax = axes[0]
ax.fill_between(daily.index, daily["q25_ntu"], daily["q75_ntu"],
                alpha=0.25, color="steelblue", label="IQR across stations")
ax.plot(daily.index, daily["median_ntu"], lw=1, color="steelblue", label="Median NTU")
for thresh, label in [(5, "low"), (15, "medium"), (30, "high")]:
    ax.axhline(thresh, ls="--", lw=0.7, color="grey")
ax.scatter(sat_coverage.index, sat_coverage["median_ntu"], s=20, zorder=5,
           color="black", label="Acquired scenes", marker="v")
ax.set_ylabel("Surface turbidity (NTU)")
ax.legend(fontsize=8)
ax.set_title("Multi-station surface turbidity with acquired satellite dates")

ax = axes[1]
for regime, color in REGIME_COLORS.items():
    mask = daily["regime"] == regime
    ax.fill_between(daily.index, 0, 1, where=mask, transform=ax.get_xaxis_transform(),
                    alpha=0.6, color=color, label=regime)
ax.scatter(sat_coverage.index, [0.5] * len(sat_coverage), s=25, color="black",
           marker="v", zorder=5, label="Acquired")
if not win_df.empty:
    for _, row in win_df.iterrows():
        ax.axvspan(row["start"], row["end"], alpha=0.35, color="yellow", zorder=3)
ax.set_ylabel("Regime")
ax.set_yticks([])
ax.legend(loc="upper right", fontsize=8, ncol=3)
ax.set_title("Turbidity regime timeline (yellow = candidate windows)")

ax = axes[2]
ax.plot(daily.index, daily["roll_cv"], lw=1, color="purple", label=f"{STABILITY_WINDOW}-day rolling CV")
ax.axhline(cv_p25, ls="--", lw=0.8, color="black", label=f"Stability threshold (p25 = {cv_p25:.2f})")
ax.fill_between(daily.index, 0, daily["roll_cv"],
                where=daily["stable"], alpha=0.3, color="green", label="Stable")
ax.set_ylabel("Coefficient of variation")
ax.legend(fontsize=8)
ax.set_title("Rolling turbidity stability")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "acquisition_date_analysis.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
x = np.arange(len(REGIME_ORDER))
w = 0.35
ax.bar(x - w/2, coverage_gap["total_days"], w, label="All CTD days",
       color=[REGIME_COLORS[r] for r in REGIME_ORDER], alpha=0.5)
ax.bar(x + w/2, coverage_gap["acquired_days"], w, label="Acquired scenes",
       color=[REGIME_COLORS[r] for r in REGIME_ORDER])
ax.set_xticks(x)
ax.set_xticklabels(REGIME_ORDER)
ax.set_ylabel("Days")
ax.legend()
ax.set_title("Regime coverage: CTD vs acquired")

ax = axes[1]
bars = ax.bar(REGIME_ORDER, coverage_gap["coverage_pct"],
              color=[REGIME_COLORS[r] for r in REGIME_ORDER])
ax.axhline(median_coverage, ls="--", color="black", label=f"Median ({median_coverage:.1f}%)")
ax.set_ylabel("Acquisition rate (%)")
ax.set_title("Fraction of days with acquired imagery per regime")
ax.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "regime_coverage_gap.png", dpi=150)
plt.show()

## 9. Export

In [ ]:
daily[["median_ntu", "q25_ntu", "q75_ntu", "coverage", "regime", "roll_cv", "stable", "acquired", "candidate"]].to_csv(
    RESULTS_DIR / "surface_turbidity_daily.csv"
)

if not win_df.empty:
    win_df.to_csv(RESULTS_DIR / "candidate_acquisition_windows.csv", index=False)
    print(f"Exported {len(win_df)} candidate windows to candidate_acquisition_windows.csv")

coverage_gap.to_csv(RESULTS_DIR / "regime_coverage_gap.csv")
print("Exported regime coverage gap to regime_coverage_gap.csv")